In [2]:
# import json
from pathlib import Path
import numpy as np
# import pytest
# import advanced_link_skdsp_v3_tx_flexible as txmod
import generate_tx_iq_dataset as genmod
import load_tx_iq_data as loadmod
from tx_controller_tone_pulse_stft_varlen import TonePulseTXControlNetVarLen, build_controlled_tone_pulse_from_variable_inputs
import torch
import math
import score_iq_decode as scorer
import argparse
import json
import tempfile
from pathlib import Path
from typing import Optional
import advanced_link_skdsp_v4_robust as link
import os

def repeat_to_length_mod(arr, target_length):
    if arr.ndim != 1:
        raise ValueError("Input array must be 1D")
    if len(arr) == 0:
        raise ValueError("Input array must not be empty")

    idx = np.arange(target_length) % len(arr)
    return arr[idx]


def delete_if_exists(filepath):
    """
    Check if a file exists and delete it.

    Parameters:
        filepath (str): Path to the file
    """
    if os.path.isfile(filepath):
        os.remove(filepath)
        print(f"Deleted: {filepath}")
    else:
        print(f"File not found: {filepath}")

# tmp_path = Path("local_test")
# out_root = tmp_path / "dataset"

# produced = genmod.generate_dataset(
#                                     output_root=out_root,
#                                     num_outputs=4,
#                                     min_total_samples=8192,
#                                     max_total_samples=12000,
#                                     section_len=1024,
#                                     num_sections=3,
#                                     seed=3,
#                                     random_payload_probability=0.5,
#                                     )

# assert len(produced) == 4

# sample_dirs = loadmod.list_sample_dirs(out_root)
# assert len(sample_dirs) == 4

# for sdir in sample_dirs:
#     assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
#     assert (sdir / "sections.npy").exists()
#     assert (sdir / "sections_meta.json").exists()

# whole = loadmod.load_whole_iq(sdir)
# sections = loadmod.load_sections(sdir)
# bundle = loadmod.load_sample_bundle(sdir)

# iq = whole["iq"]
# whole_meta = whole["meta"]
# secs = sections["sections"]
# sections_meta = sections["meta"]

# assert iq.dtype == np.complex64
# assert secs.dtype == np.complex64
# assert secs.shape == (3, 1024)

# assert whole_meta["actual_num_samples"] == len(iq)
# assert sections_meta["whole_num_samples"] == len(iq)
# assert len(sections_meta["starts"]) == 3

# for idx, start in enumerate(sections_meta["starts"]):
#     np.testing.assert_array_equal(secs[idx], iq[start:start + 1024])

# assert bundle["whole_iq"].shape == iq.shape
# assert bundle["sections"].shape == secs.shape



In [2]:
train_path = Path("train_set_00")
test_path = Path("test_set_00")
val_path = Path("val_set_00")

for _ in [train_path, test_path, val_path]:
    out_root = _ / "dataset"
    produced = genmod.generate_dataset(
                                        output_root=out_root,
                                        num_outputs=100,
                                        min_total_samples=20000,
                                        max_total_samples=100000,
                                        section_len=1024,
                                        num_sections=3,
                                        seed=3,
                                        random_payload_probability=0.0,
                                        )

In [3]:
# "C:\Users\theon\Jamming\train_set_00"
train_path_dat = Path("C:/Users/theon/Jamming/train_set_00/dataset")
sample_dirs = loadmod.list_sample_dirs(train_path_dat)
sdir = sample_dirs[0]
whole = loadmod.load_whole_iq(sdir)
# for sdir in sample_dirs:
#     assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
#     assert (sdir / "sections.npy").exists()
#     assert (sdir / "sections_meta.json").exists()

# whole = loadmod.load_whole_iq(sdir)
# sections = loadmod.load_sections(sdir)

In [ ]:
sections = loadmod.load_sections(sdir)
sections['sections'][0].shape

In [4]:
whole['iq'].shape[0]

123040

In [9]:
whole

{'iq': array([-0.00661431-0.00238922j, -0.00341755+0.00149117j,
        -0.00124424-0.00218887j, ...,  0.28430367-0.2098175j ,
         0.2830943 -0.20910673j,  0.28348896-0.21429396j],
       shape=(123040,), dtype=complex64),
 'meta': {'payload_source': 'message',
  'message_length': 776,
  'message': '8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1L10e0tza9 3Ce6 KRDiKpFfe z3gb9pv MEc0goRqG9hTCzppvEJqa ZgIFI 2g8Pahqfpgi9el   OuOjSXXMUHyQ2pqaWkwfV6Wf3gV O116cFBIj2C2qdPVHp7pWnOna8sEXflmWwdRpdo9KMle EnmhPxjExF6YGVYS1kW E0u1 9tZtum 9 4xrIzs ODL0IBNcI9WzwUEP9s19A7d9jZf7DEiBGEyHrxeGvfOx2lkplHLeT5RadQQ3W jA YqOpJj3o4GmVV5LHnRo  ThLxtwV6pnsQ1Mx69NwinxYTmIIXgrf9 IF TQZ5iT otImooxy1YqsYyvwzGVLd40XON M9dyanD w6zyBe 4oKtr7lgdUD j cRPQSrkekRAiz3C Onf0jzuY 8i2A Mc76Z4x6eGUV5UZCaAHVs6yuAcvZ vdrov4 xhcZ5O0egEZfY dCEmX8yvQoSpgLJ7M FIdRSOlh3landlcv e9gy Qz9R9SeWNYlLx0o XQZwXTxU14D49S

In [51]:
sdir_01 = sample_dirs[1]
whole_01 = loadmod.load_whole_iq(sdir_01)
whole_01


{'iq': array([-7.0849201e-03-2.0398074e-03j,  9.4907358e-05+1.9195544e-03j,
        -2.1207645e-03-1.0466345e-04j, ..., -5.6718517e-02+1.6181794e-01j,
        -2.5296977e-02+8.1467651e-02j, -1.9126255e-03+9.7739976e-04j],
       shape=(34541,), dtype=complex64),
 'meta': {'payload_source': 'message',
  'message_length': 983,
  'message': '4WAoa7MjRSy jU2i BFShQE34kGBPvAB71Vy0908eLcxmdtL  h8fyAJ 3eS7zKsn4M04jAt KWVu3N78CUKKd7VTMF bbqtcvgaA7TUe xEbJ3Rgm5NHF 1HRfd2evFr0 RrJcvfc h67  1V vLxjrn0T46JG5KtOr e0 D6IdOo qJIn3jVe  5yNSxXYOgIBeOOYKeq1G0k DzkopaKi3I97ILgwE vstw7YbsYgxwNyqsgtBWn3XxdJnqosLqXTjyaVs9FiT nO8cS407N4t6XzLwMvOIzqgh0wobupZVJl74PsBP6 VOTSRKMIxp CRFK3I5q9Qw 4gi12LhEWXBjU Bh p3XUbMV VZ4Vn sPCaViarkAP3KzddO4UB4Rp WC8rNKypwk3dSWa x8JrXA 3M3Kmki4PibP92mQ1CPzYlg d EkxD8YKS6DzMRrU TOA5h3Ny Bxn6p YBOxeP FXhEZUXCwKGSHgRoACM4wvF9ABUDwY2wM4Vhk6 cwI 1 CX  SZ9wo1RsPnSsM9t5V fSzlMKP0Lb2S29VxFLyXoPuO wY7z0Unh9zv vpmpX OYJi9JODHpBo9V3sHVxH zCFHSRnmrkC6QoPsvf I rV42 23GLIXtfev3aBq AVGfTzpuBj

In [32]:
whole['meta']['sample_rate_hz']

500000.0

In [3]:
device = "cuda"
model = mod.TXControlNetSTFT3Input(in_ch=5, base_ch=16, n_scalar_features=15).to(device)

In [4]:
train_path_dat = Path("train_set_00/dataset")
sample_dirs = loadmod.list_sample_dirs(train_path_dat)
sdir = sample_dirs[0]

sections = loadmod.load_sections(sdir)

iq1 = sections['sections'][0]
iq2 = sections['sections'][1]
iq3 = sections['sections'][2]

sampling_freq = float(2e9)

feat1, info1 = mod.preprocess_single_iq(iq1, sampling_freq)
feat2, info2 = mod.preprocess_single_iq(iq2, sampling_freq)
feat3, info3 = mod.preprocess_single_iq(iq3, sampling_freq)

x1 = torch.from_numpy(feat1).unsqueeze(0).to(device)
x2 = torch.from_numpy(feat2).unsqueeze(0).to(device)
x3 = torch.from_numpy(feat3).unsqueeze(0).to(device)

scalar_side = torch.tensor(
    [[
        math.log10(1_000_000.0) / 10.0,
        info1["digital_center_est_hz"] / 1_000_000.0,
        info2["digital_center_est_hz"] / 1_000_000.0,
        info3["digital_center_est_hz"] / 1_000_000.0,
        np.mean([info1["mad_sigma"], info2["mad_sigma"], info3["mad_sigma"]]),
        np.mean([info1["dc_i"], info2["dc_i"], info3["dc_i"]]),
        np.mean([info1["dc_q"], info2["dc_q"], info3["dc_q"]]),
        np.mean([info1["rx_input_power"], info2["rx_input_power"], info3["rx_input_power"]]),
        info1["peak_hz"] / 1_000_000.0,
        info2["peak_hz"] / 1_000_000.0,
        info3["peak_hz"] / 1_000_000.0,
        info1["centroid_hz"] / 1_000_000.0,
        info2["centroid_hz"] / 1_000_000.0,
        info3["centroid_hz"] / 1_000_000.0,
        np.mean([info1["rf_center_est_hz"], info2["rf_center_est_hz"], info3["rf_center_est_hz"]]) / 1_000_000.0,
    ]],
    dtype=torch.float32,
    device=device,
)

y = model(x1, x2, x3, scalar_side)

In [5]:
y

{'fec_logits': tensor([[ 0.8558,  0.2076, -0.2347]], device='cuda:0',
        grad_fn=<AddmmBackward0>),
 'noise_color_logits': tensor([[-0.0803, -0.1194,  0.9415,  0.3098,  0.8838]], device='cuda:0',
        grad_fn=<AddmmBackward0>),
 'fading_mode_logits': tensor([[ 0.7498, -0.2353,  0.3447, -0.1407]], device='cuda:0',
        grad_fn=<AddmmBackward0>),
 'burst_color_logits': tensor([[ 0.4411,  0.1308, -0.0459,  0.0723, -0.2380]], device='cuda:0',
        grad_fn=<AddmmBackward0>),
 'interleave_logit': tensor([[-0.0457]], device='cuda:0', grad_fn=<AddmmBackward0>),
 'sample_rate_scale': tensor([[1.6357]], device='cuda:0', grad_fn=<AddBackward0>),
 'rf_center_delta_hz': tensor([[-81659.8672]], device='cuda:0', grad_fn=<MulBackward0>),
 'carrier_hz_norm': tensor([[0.2682]], device='cuda:0', grad_fn=<TanhBackward0>),
 'snr_db': tensor([[21.5942]], device='cuda:0', grad_fn=<MulBackward0>),
 'fading_block_len_norm': tensor([[0.6347]], device='cuda:0', grad_fn=<SigmoidBackward0>),
 'rician

In [17]:
type(jam_iq['tx_iq'])

numpy.ndarray

In [5]:
device = "cuda"

epochs = 2
jammer_sampling_freq = 2e9
scoring_dict = {}

model = TonePulseTXControlNetVarLen(in_ch=4, base_ch=16, n_scalar_features=6, max_tones=8).to(device)
   
for epoch in range(epochs):
    epoch_score = 0.0
    print(f'epoch : {epoch}')
    sample_dirs = loadmod.list_sample_dirs(train_path_dat)
    for sdir in sample_dirs:
        whole = loadmod.load_whole_iq(sdir)
        
        whole_iq = whole['iq']
        sections = loadmod.load_sections(sdir)
        iq1 = sections['sections'][0]
        iq2 = sections['sections'][1]
        iq3 = sections['sections'][2]
        whole_sample_rate = whole['meta']['sample_rate_hz']
        print('resampling 1')
        iq1 = link.resample_iq(iq1, whole_sample_rate, jammer_sampling_freq)[:1024]
        print('resampling 2')
        iq2 = link.resample_iq(iq2, whole_sample_rate, jammer_sampling_freq)[:1024]
        print('resampling 3')
        iq3 = link.resample_iq(iq3, whole_sample_rate, jammer_sampling_freq)[:1024]
        
        print('jam_iq')
        jam_iq =  build_controlled_tone_pulse_from_variable_inputs(
                                                                    model=model,
                                                                    rx_iq_windows=[iq1, iq2, iq3],
                                                                    intake_sample_rate_hz=jammer_sampling_freq,
                                                                    desired_output_iq_len=100_000,
                                                                    user_peak_power_fraction=0.1,
                                                                    seed=11,
                                                                    device=device,
                                                                )

        print('jam resampling')

        jam_iq_rx_resam = link.resample_iq(jam_iq['tx_iq'], jammer_sampling_freq, whole_sample_rate)
        jam_iq_rx_resam = repeat_to_length_mod(jam_iq_rx_resam, whole_iq.shape[0])
        jammed = whole_iq + jam_iq_rx_resam[:whole_iq.shape[0]]
        jam_path = sdir / "jammed_iq.npy"
        meta_path = sdir / "whole_meta.json"
        score = 0.0
        try:
            rx_result = link.rx_command_iq(jammed, whole["meta"])
            score = scorer.score_decode(rx_result, whole["meta"])
        except RuntimeError as e:
            estr = e.__str__() 
            if estr == "No valid packet found after acquisition, header decode, FEC decode, and CRC":
                print(estr)
                pass
            else:
                raise e
        epoch_score += score
        print(f'sdir : {sdir} score: {score}')


    scoring_dict[epoch] = epoch_score

epoch : 0
resampling 1
resampling 2
resampling 3
jam_iq


TypeError: build_tone_pulse_iq_object() got an unexpected keyword argument 'carrier_hz'

In [9]:
whole["meta"]

{'payload_source': 'message',
 'message_length': 776,
 'message': '8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1L10e0tza9 3Ce6 KRDiKpFfe z3gb9pv MEc0goRqG9hTCzppvEJqa ZgIFI 2g8Pahqfpgi9el   OuOjSXXMUHyQ2pqaWkwfV6Wf3gV O116cFBIj2C2qdPVHp7pWnOna8sEXflmWwdRpdo9KMle EnmhPxjExF6YGVYS1kW E0u1 9tZtum 9 4xrIzs ODL0IBNcI9WzwUEP9s19A7d9jZf7DEiBGEyHrxeGvfOx2lkplHLeT5RadQQ3W jA YqOpJj3o4GmVV5LHnRo  ThLxtwV6pnsQ1Mx69NwinxYTmIIXgrf9 IF TQZ5iT otImooxy1YqsYyvwzGVLd40XON M9dyanD w6zyBe 4oKtr7lgdUD j cRPQSrkekRAiz3C Onf0jzuY 8i2A Mc76Z4x6eGUV5UZCaAHVs6yuAcvZ vdrov4 xhcZ5O0egEZfY dCEmX8yvQoSpgLJ7M FIdRSOlh3landlcv e9gy Qz9R9SeWNYlLx0o XQZwXTxU14D49SIv X ftvc7lmOEhg57QVajyZnRNo5kAEgtsboDKAC 1 Oy7wkfodmzGkn7ZCn SZ5oL4WAoa7MjRSy jU2',
 'random_bits': None,
 'random_seed': None,
 'payload_len': 776,
 'payload_len_bytes': 776,
 'target_num_samples': None,
 'actual_num_samples': 123040,
 '

In [ ]:
def compute_returns(rewards, gamma: float):
    """
    Compute discounted returns for one episode.
    """
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns


def train_policy_gradient(
    env,
    policy: nn.Module,
    optimizer: optim.Optimizer,
    num_episodes: int = 1000,
    gamma: float = 0.99,
    device: str = "cpu",
    max_steps_per_episode: int = 1000,
    normalize_returns: bool = True,
    print_every: int = 50,
):
    """
    Generic REINFORCE training loop.

    Args:
        env: Gym-like environment with reset() and step(action)
        policy: PyTorch model mapping observation -> action logits
        optimizer: PyTorch optimizer
        num_episodes: number of training episodes
        gamma: discount factor
        device: torch device
        max_steps_per_episode: safety cap per episode
        normalize_returns: whether to normalize discounted returns
        print_every: logging frequency
    """
    device = torch.device(device)
    policy.to(device)

    reward_history = []

    for episode in range(1, num_episodes + 1):
        # Gymnasium reset API compatibility
        reset_result = env.reset()
        state = reset_result[0] if isinstance(reset_result, tuple) else reset_result

        log_probs = []
        rewards = []
        total_reward = 0.0

        for step in range(max_steps_per_episode):
            action, log_prob = select_action(policy, state, device)

            step_result = env.step(action)
            if len(step_result) == 5:
                next_state, reward, terminated, truncated, _ = step_result
                done = terminated or truncated
            else:
                next_state, reward, done, _ = step_result

            log_probs.append(log_prob)
            rewards.append(reward)
            total_reward += reward
            state = next_state

            if done:
                break

        # Compute discounted returns
        returns = compute_returns(rewards, gamma)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        if normalize_returns and len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Policy gradient loss
        loss = []
        for log_prob, G in zip(log_probs, returns):
            loss.append(-log_prob * G)
        loss = torch.stack(loss).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reward_history.append(total_reward)

        if episode % print_every == 0:
            avg_reward = sum(reward_history[-print_every:]) / print_every
            print(
                f"Episode {episode:4d} | "
                f"Avg Reward: {avg_reward:8.3f} | "
                f"Last Reward: {total_reward:8.3f} | "
                f"Loss: {loss.item():8.3f}"
            )

    return reward_history